# Notebook 13.1  A mispronunciation-detection prototype with Goodness of Pronunciation and a neural score

**Goal.** Force-align learner audio to a target, compute a per-phone Goodness of Pronunciation (GOP) and a neural mispronunciation score, flag likely errors, and compare with expert annotations, then discuss how the same pipeline supports a tajwīd or clinical use case.

**What runs here.** The GOP computation, the flagging, and the agreement metrics are real and run with no downloads. Forced alignment and the acoustic-model posteriors are represented by provided per-phone posterior tables (a synthetic fallback), so the notebook always runs; the markdown says where a real aligner and acoustic model would plug in. This accompanies Chapter 13. GOP is a screening score: it flags suspicious phones, while diagnosis needs error-labeled data.

## 1. Setup

In [ ]:
import math
from collections import Counter
EPS = 1e-8   # avoid log(0)
print('ready')

## 2. A forced alignment with acoustic-model posteriors

In a real system, an aligner (for example the Montreal Forced Aligner) segments the learner audio into the target phones, and an acoustic model gives, for each segment, the posterior probability of the intended phone. Here each utterance is a list of (intended_phone, posterior_of_intended_phone, is_error) records; `is_error` is the expert label used later for evaluation.

**To use real audio:** align with an aligner, run the acoustic model to get frame posteriors, average the posterior of the intended phone over its segment, and replace the `posterior` field below.

In [ ]:
# target word رحيم /r a ḥ ii m/; the learner replaced ḥ with h
utterances = [
  # (utt_id, [(phone, posterior_of_intended, expert_is_error)])
  ('u1', [('r',0.93,0),('a',0.88,0),('ḥ',0.20,1),('ii',0.81,0),('m',0.90,0)]),
  ('u2', [('s',0.85,0),('a',0.80,0),('l',0.78,0),('aa',0.30,1),('m',0.86,0)]),
  ('u3', [('k',0.91,0),('t',0.87,0),('aa',0.84,0),('b',0.40,1)]),
  ('u4', [('b',0.92,0),('a',0.89,0),('y',0.83,0),('t',0.88,0)]),  # all correct
]
print('utterances:', len(utterances))

## 3. Goodness of Pronunciation

A common GOP form is the negative log of the posterior of the intended phone (optionally length-normalized): a high posterior gives a low score, a low posterior gives a high score. We add an epsilon inside the log to avoid log(0).

In [ ]:
def gop(posterior):
    return -math.log(posterior + EPS)

for uid, phones in utterances:
    scores = [(ph, round(gop(p),2)) for ph,p,_ in phones]
    print(uid, scores)

## 4. Flag mispronunciations and tune the threshold

A phone is flagged when its GOP exceeds a threshold. The threshold is tuned on this (expert-labeled) data by trying candidate values and picking the one with the best detection trade-off; in practice tune it on a held-out validation set, per phone where possible.

In [ ]:
flat = [(ph, gop(p), err) for _,phones in utterances for ph,p,err in phones]

def detection_scores(thr):
    tp=fp=fn=tn=0
    for ph,g,err in flat:
        flagged = g > thr
        if flagged and err: tp+=1
        elif flagged and not err: fp+=1
        elif not flagged and err: fn+=1
        else: tn+=1
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    return prec,rec,f1

best=None
for thr in [0.5,0.8,1.0,1.2,1.5,2.0]:
    p,r,f = detection_scores(thr)
    print(f'thr={thr:.1f}  precision={p:.2f} recall={r:.2f} F1={f:.2f}')
    if best is None or f>best[1]: best=(thr,f)
print('\nbest threshold by F1:', best[0])

## 5. Agreement with expert annotations

Treat the system's flags at the chosen threshold as one rater and the expert labels as another, and compute Cohen's kappa to see how well the system agrees with the human ceiling. Kappa corrects for chance agreement.

In [ ]:
def cohen_kappa(a, b):
    n=len(a); po=sum(1 for x,y in zip(a,b) if x==y)/n
    # expected agreement from marginal rates
    pa1=sum(a)/n; pb1=sum(b)/n
    pe=pa1*pb1 + (1-pa1)*(1-pb1)
    return (po-pe)/(1-pe) if pe<1 else 1.0

thr=best[0]
sys_flags=[1 if gop(p)>thr else 0 for _,phones in utterances for _,p,_ in phones]
expert   =[err for _,phones in utterances for _,_,err in phones]
p,r,f=detection_scores(thr)
print(f'at threshold {thr}: precision={p:.2f} recall={r:.2f} F1={f:.2f}')
print('Cohen kappa (system vs expert): %.2f' % cohen_kappa(sys_flags, expert))

## 6. From detection to diagnosis, and to other use cases

GOP told us *which* phones are suspect, not *what* went wrong. Diagnosis (for example, that ḥ became h) needs a model trained on annotated learner errors, or a recognizer that can output the actually-produced phone. The same pipeline generalizes:

- **Tajwīd:** replace the phone set and rules with a recitation phonetic script that encodes elongation and assimilation, and score against tajwīd-correct targets, with expert validation.
- **Clinical / disordered speech:** the same alignment-and-score idea can track a patient's articulation over time, but it must be personalized to the speaker and validated clinically, never used to label a speech difference as an error.

## 7. Where to go next

- Plug in a real aligner and acoustic model to replace the provided posteriors; compute length-normalized GOP from frame posteriors.
- Add a neural mispronunciation-detection model trained on annotated errors for diagnosis, and compare its agreement with experts against GOP's.
- Tune thresholds per phone on a held-out validation set, and report false alarms on legitimate dialectal variation separately.
- Apply strict consent and privacy governance for any learner, child, or clinical data.